# Silver Layer - CRM Customers

This notebook transforms `workspace.bronze.crm_cust_info_raw` into a cleaned Silver table: `workspace.silver.crm_customers`.

## Tools and Operations
- **PySpark DataFrame API** for transformations
- **String standardization** (trim, empty-to-null, value mapping)
- **Data quality checks** (null/duplicate inspection)
- **Delta table write** for downstream Gold modeling

## Output
- `workspace.silver.crm_customers`


## 1) Setup


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *
from pyspark.sql import DataFrame
from itertools import chain
from pyspark.sql.window import Window

## 2) Load and Inspect Source Data


In [0]:
df_raw = spark.table("workspace.bronze.crm_cust_info_raw")

In [0]:
df_raw.limit(100).display()

In [0]:
print(df_raw.count())
df_raw.printSchema()

## 3) Transformations


### Rename Columns


In [0]:
name_map = {
    'cst_id' : 'customer_id',
    'cst_key' : 'customer_key',
    'cst_firstname' : 'first_name',
    'cst_lastname' : 'last_name',
    'cst_marital_status' : 'marital_status', 
    'cst_gndr' : 'gender', 
    'cst_create_date' : 'create_date'   
}

df = df_raw.select([F.col(c).alias(name_map.get(c, c)) for c in df_raw.columns])

In [0]:
df.columns

### Trim String Columns


In [0]:
df = df.select([
    F.when(F.trim(F.col(f.name)) == "", F.lit(None)).otherwise(F.trim(F.col(f.name))).alias(f.name)
    if isinstance(f.dataType, StringType) 
    else F.col(f.name)
    for f in df.schema.fields
])

df.display()

### Normalize Encoded Values


In [0]:
df.groupBy("marital_status").count().display()
df.groupBy("gender").count().display()

In [0]:
def map_abbrev_column(df: DataFrame, str_col: str, mapping: dict) -> DataFrame:
    mapping = {k.strip().lower(): v for k, v in mapping.items()}
    map_expr = F.create_map([F.lit(x) for x in chain(*mapping.items())])
    key = F.lower(F.trim(F.col(str_col)))
    return df.withColumn(str_col, F.coalesce(map_expr[key], key))

marital_map = {"m": "Married", "s": "Single"}
gender_map = {"m": "Male", "f": "Female"}

df = map_abbrev_column(df, "marital_status", marital_map)
df = map_abbrev_column(df, "gender", gender_map)
display(df)

### Null Checks


In [0]:
df.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
]).display()

In [0]:
df = df.filter(F.col("customer_id").isNotNull())

display(
    df.filter(
        (F.col("first_name").isNull()) |
        (F.col("last_name").isNull()) |
        (F.col("marital_status").isNull())
    )
)

### De-duplication


In [0]:
print(df.count())
print(df.select("customer_id").distinct().count())
print(df.select("customer_key").distinct().count())

In [0]:
dup_ids = (df
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .select("customer_id")
)

df_dups = df.join(dup_ids, on="customer_id", how="inner")
display(df_dups)

In [0]:
cols = ["first_name", "last_name", "marital_status", "gender"]

nonnull_cnt = df.withColumn(
    "_nonnull_cnt",
    sum(
        F.when(F.col(c).isNotNull(), F.lit(1)).otherwise(F.lit(0))
        for c in cols
    )
)

w = Window.partitionBy("customer_key").orderBy(F.col("_nonnull_cnt").desc())

df = (
    nonnull_cnt
    .withColumn("_rn", F.row_number().over(w))
    .filter(F.col("_rn") == 1)
    .drop("_nonnull_cnt", "_rn")
)

df.display()
print(df.count())
print(df.select("customer_id").distinct().count())
print(df.select("customer_key").distinct().count())

## 4) Write Silver Table


In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_customers")

## 5) Validation


In [0]:
%sql
SELECT * FROM workspace.silver.crm_customers LIMIT 50